In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np

from mdx2.io import loadobj
from mdx2.report.map_statistics import calc_correlation_of_half_datasets, calc_intensity_statistics, prepare_table

# Map statistics

Compute statistics for merged diffuse map

## Parameters

In [ ]:
# These metadata fields are injected by mdx2.report, regardless of the template. Don't change this line.
_metadata: dict = {}

# Notebook parameters
#input_files: list[str] = None # list of input file paths: required
crystal_source: str = None  # (nexus_file_path:object name)
symmetry_source: str = None  # (nexus_file_path:object name)
hkl_table_source: str = None  # (nexus_file_path:object name)

bin_width: float = 0.01  # bin width for statistics vs s

In [ ]:
# pretty-print the metadata and parameters
pprint({
    "metadata": _metadata,
    "parameters": {"crystal_source": crystal_source,
                   "symmetry_source": symmetry_source,
                   "hkl_table_source": hkl_table_source,
                   "bin_width": bin_width,
                   },
})

## Results

In [ ]:
crystal = loadobj(*crystal_source.split(":"))
symmetry = loadobj(*symmetry_source.split(":"))
hkl_table = loadobj(*hkl_table_source.split(":"))

### Map properties

In [ ]:
def _calc_unit_cell(ub):
    a, b, c = ub[:, 0], ub[:, 1], ub[:, 2]
    a_len = np.linalg.norm(a)
    b_len = np.linalg.norm(b)
    c_len = np.linalg.norm(c)
    alpha = np.arccos(np.dot(b, c) / (b_len * c_len)) * 180 / np.pi
    beta = np.arccos(np.dot(a, c) / (a_len * c_len)) * 180 / np.pi
    gamma = np.arccos(np.dot(a, b) / (a_len * b_len)) * 180 / np.pi
    return a_len, b_len, c_len, alpha, beta, gamma

print("Laue group:", symmetry.laue_group_symbol)
a, b, c, alpha, beta, gamma = _calc_unit_cell(crystal.ub_matrix)
print("Reciprocal cell dimensions")
print("   a*, b*, c* (Å⁻¹): {:.5f}, {:.5f}, {:.5f}".format(a, b, c))
print("     α*, β*, γ* (°): {:.2f}, {:.2f}, {:.2f}".format(alpha, beta, gamma))
print("Reciprocal cell volume (Å⁻³): {:.3e}".format(np.linalg.det(crystal.ub_matrix)))
print("Voxel subdivisions:", hkl_table.ndiv)
print("Voxel volume (Å⁻³): {:.3e}".format(np.linalg.det(crystal.ub_matrix) / np.prod(hkl_table.ndiv)))

In [ ]:
df = prepare_table(hkl_table, crystal, symmetry, bin_width=bin_width)
df.describe().T

### Statistics vs. resolution

In [ ]:
# compute the statistics with and without voxels centered on Bragg reflections (strong halos can dominate stats)
df_isoavg_all = calc_intensity_statistics(df, bin_width=bin_width, exclude_bragg=False)
df_isoavg_excluded = calc_intensity_statistics(df, bin_width=bin_width, exclude_bragg=True)

def _set_tight_limits(df, ax):
    df = df.drop(columns=['s'])
    col_mins = df.where(df > 0).min()
    col_maxs = df.max()
    vmin = col_mins.min()
    vmax = col_maxs.max()
    ax.set_ylim(vmin*0.5, vmax*2)

fig, (ax0, ax1) = plt.subplots(1, 2, sharey=True, figsize=(8, 4))

df_isoavg_all.set_index('s').plot(logy=True, ax=ax0, title="All voxels", legend=False)
ax0.set_ylabel("Diffuse intensity (ph s$^{-1}$ sr$^{-1}$)")

df_isoavg_excluded.set_index('s').plot(logy=True, ax=ax1, title="Near-Bragg data excluded", legend=False)
_set_tight_limits(df_isoavg_excluded.iloc[5:-5], ax1) # exclude the first and last 5 bins when setting limits.

[ax.set_xlabel("1/d (Å$^{-1}$)") for ax in (ax0, ax1)]
fig.legend(*ax1.get_legend_handles_labels(), loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### Correlelation between half-datasets

In [ ]:
# compute the statistics with and without voxels centered on Bragg reflections (strong halos can dominate stats)
if "group_0_intensity" not in df.columns:
    print("The hkl_table does not contain split intensities, skipping correlation plot")
else:
    df_cc_all = calc_correlation_of_half_datasets(df, bin_width=bin_width, exclude_bragg=False)
    df_cc_excluded = calc_correlation_of_half_datasets(df, bin_width=bin_width, exclude_bragg=True)
    colors = ["black", "gray"]

    fig, (ax0, ax1) = plt.subplots(1, 2, sharey=True, figsize=(8, 3.5))
    df_cc_all.set_index('s').plot(ax=ax0, title="All voxels", legend=False, color=colors)
    ax0.set_ylabel("Correlation of half datasets (CC 1/2)")

    df_cc_excluded.set_index('s').plot(ax=ax1, title="Near-Bragg data excluded", legend=False, color=colors)
    ax1.set_ylim(0, 1)

    [ax.set_xlabel("1/d (Å$^{-1}$)") for ax in (ax0, ax1)]
    fig.legend(*ax1.get_legend_handles_labels(), loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()